In [11]:
cd DRAFT

/home2/rudra.dhar/DRAFT


In [2]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset
from dotenv import load_dotenv

2025-09-16 20:19:17.772751: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-16 20:19:17.791350: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-16 20:19:17.796909: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-16 20:19:17.811677: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-16 20:19:19.696677: W tensorflow/compiler/tf2

In [12]:
model_name = "google/gemma-3-4b-it"
cache_dir = "/scratch/rudra.dhar/cache"
output_dir = "/scratch/rudra.dhar/results"

load_dotenv()
HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE")

In [26]:
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir, token=HUGGINGFACE_TOKEN)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    device_map="auto",
    torch_dtype=torch.float16,
    token=HUGGINGFACE_TOKEN
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [27]:
# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()

In [28]:
data_files = {"train": "Retrieval/CDtrain.jsonl", "validation": "Retrieval/CDval.jsonl"}
dataset = load_dataset("json", data_files=data_files)

In [29]:
dataset['train'] = dataset['train'].shuffle(seed=42).select(range(100))  # For quick testing, use only 1000 samples
dataset['validation'] = dataset['validation'].shuffle(seed=42).select(range(10))  # For quick testing, use only 100 samples

In [30]:
def format_example(example):
    context = example["Anchor"]["Context"]
    decision = example["Anchor"]["Decision"]

    # Turn into chat-like input
    messages = [
        {"role": "system", "content": "You are an expert software architect responsible for maintaining and thoroughly documenting all architectural decisions. You are writing an Architectural Decision Record for a software. Give a ## Decision corresponding to the ## Context provided by the User. Provide only the Decision in about 2-400 words. Do not add any explanations, introductions, or additional responses."},
        {"role": "user", "content": f"## Context: {context}"},
        {"role": "assistant", "content": f"## Decision: {decision}"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

# the formated text is stored in the "text" field
dataset = dataset.map(format_example)


In [31]:
# Tokenize
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=1024
    )

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [34]:
# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Training arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=2,
    warmup_steps=20,
    weight_decay=0.01,
    fp16=True,
    bf16=False,
    save_total_limit=2,
    report_to="none",
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator
)

In [ ]:
trainer.train()

/home2/rudra.dhar/miniconda3/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


In [6]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.backends.cudnn.version())
print(torch.cuda.is_available())


2.7.1+cu118
11.8
90100
False


In [5]:
!unset CUDA_VISIBLE_DEVICES